# OptiSystem 25 Gb/s PAM4 — reproducible statistical Q analysis

This notebook reproduces the **nine** unique TX/RX waveform conditions, the **five-point fiber sweep**, and the **five-point receiver-bandwidth sweep**. Source trace SHA-256 hashes and the previously reported Q values are verified.

**Not OptiSystem built-in Q; not BER.** PAM4's minimum adjacent-level statistical separation is computed from known TX reference symbols and corresponding RX amplitudes, after choosing a receiver sampling phase with the first-half training data. The full-data and held-out estimates are both saved.


## 1. Get the project data

After uploading the repository, leave `SOURCE_MODE = 'github'` and click **Run**. If GitHub is not yet populated, switch to `SOURCE_MODE = 'zip'` and upload the full ready-to-upload ZIP when prompted. Local Jupyter users may run the notebook from within the extracted project folder.


In [ ]:
from pathlib import Path
import subprocess, sys, zipfile

SOURCE_MODE = 'github'  # choose 'github' or 'zip' (manually upload project ZIP in Colab)
REPO_URL = 'https://github.com/SHAHRIAR-HOSSAIN-RIMON/OptiSystem_lab_final'

# Detect an existing local checkout first.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data'/'manifest.csv').exists()), None)
if ROOT is None:
    if SOURCE_MODE == 'github':
        ROOT = Path('/content/OptiSystem_lab_final')
        if not (ROOT/'data'/'manifest.csv').exists():
            subprocess.run(['git','clone','--depth','1',REPO_URL,str(ROOT)],check=True)
    else:
        from google.colab import files
        uploaded = files.upload()
        zip_name = next(k for k in uploaded if k.endswith('.zip'))
        dest = Path('/content/pam4_project')
        dest.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_name) as z:
            z.extractall(dest)
        candidates = list(dest.rglob('data/manifest.csv'))
        if len(candidates) != 1:
            raise RuntimeError('Could not uniquely locate data/manifest.csv in ZIP')
        ROOT = candidates[0].parents[1]
ROOT = ROOT.resolve()
print('Project directory:', ROOT)
assert (ROOT/'data'/'manifest.csv').exists(), 'Missing project data; upload the repository files first.'


## 2. Run independent verification and regenerate outputs

This runs the same Python implementation for every condition and verifies source hashes and numerical results before generating figures. The input waveform values are not regenerated or altered.


In [ ]:
subprocess.run([sys.executable, str(ROOT/'scripts'/'reproduce_all.py'), '--root', str(ROOT)], check=True)


## 3. Inspect the two data tables and graphs


In [ ]:
import csv
from IPython.display import display, Image

def show_csv(path, x_key):
    with open(path, newline='', encoding='utf-8') as f:
        r = list(csv.DictReader(f))
    print(f'{x_key:>13} | Estimated min Q | Held-out min Q')
    print('-'*52)
    for row in r:
        print(f'{row[x_key]:>13} | {float(row["q_min"]):>15.6f} | {float(row["q_min_holdout"]):>14.6f}')

show_csv(ROOT/'results'/'summary'/'fiber_sweep.csv', 'fiber_km')
display(Image(filename=str(ROOT/'results'/'figures'/'fiber_sweep_Q.png')))
show_csv(ROOT/'results'/'summary'/'bandwidth_sweep.csv', 'cutoff_GHz')
display(Image(filename=str(ROOT/'results'/'figures'/'bandwidth_sweep_Q.png')))


## 4. Inspect an individual PAM4 run's four level populations

The level means and standard deviations are from RX waveform samples labelled by the corresponding known TX symbols. No bit-error rate is computed.


In [ ]:
with open(ROOT/'results'/'per_run'/'fiber_10km_7p5GHz_levels.csv', newline='', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        print(row)


## Limitations and reproducibility

- **1024 bits**, 32 samples per bit, 512 PAM4 symbols, 496 retained after edge exclusion. These waveform estimates do **not** determine rare-event BER.
- The timing offset is chosen by maximizing first-half minimum Q; therefore reporting the **full-sample Q** is partly in-sample. The **second-half minimum Q** is also provided as a diagnostic, not a statistically independent experiment.
- Separate OptiSystem calculations may use distinct PRBS sequences. The results support comparisons at the tested settings but do not quantify statistical uncertainty or prove a universal optimum.
- The RX level spread includes noise, intersymbol interference, and other waveform distortions. The equation here is a level separation metric, **not** the output of an OptiSystem BER analyzer.

To explain to a teacher: *“I exported the PAM4 TX and RX waveforms from the same OptiSystem run, aligned the receiver sampling times, grouped RX samples by known transmitted level, calculated adjacent-level separation divided by the sum of standard deviations, and reported the minimum over the three PAM4 eyes. The notebook reproduces each number and explicitly reports that this is a waveform-derived statistical Q estimate, not measured BER.”*
